# 1. Library imports

In [ ]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Data Collection & Cleaning


## 2.1 Data Collection

In [ ]:
## collecting data

# dataset was formed with SQL query and exported in csv
# due to limited access to BigQuery storage API

df = pd.read_csv(
    "/Users/kovalskyy/Downloads/AB  Test Analysis/data sources/raw_ab_test_data.csv"
    )

df.head()

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,3978035233,Norway,mobile,Europe,Direct,2,2,session with orders,1
1,2020-11-04,983674432,Guatemala,desktop,Americas,Undefined,2,1,session with orders,1
2,2020-11-05,6992404788,Lebanon,desktop,Asia,Paid Search,2,1,session with orders,1
3,2020-11-08,9642632950,Belgium,desktop,Europe,Paid Search,2,1,session with orders,1
4,2020-11-13,539142486,Peru,desktop,Americas,Paid Search,2,1,session with orders,1


## 2.2 Data Cleaning

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93280 entries, 0 to 93279
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   date           93280 non-null  str  
 1   ga_session_id  93280 non-null  int64
 2   country        93280 non-null  str  
 3   device         93280 non-null  str  
 4   continent      93280 non-null  str  
 5   channel        93280 non-null  str  
 6   test           93280 non-null  int64
 7   test_group     93280 non-null  int64
 8   event_name     93280 non-null  str  
 9   value          93280 non-null  int64
dtypes: int64(4), str(6)
memory usage: 7.1 MB


In [10]:
# cheking number of Nan values
print(f"Nan values by column:\n{df.isna().sum()}")

Nan values by column:
date             0
ga_session_id    0
country          0
device           0
continent        0
channel          0
test             0
test_group       0
event_name       0
value            0
dtype: int64


In [11]:
# cheking if there are duplicated rows
print(f"Number of duplicated rows: {df.duplicated().sum()}")

Number of duplicated rows: 0


In [12]:
# cheking if all session_ids are unique
print(f"Total session_id rows: {df.shape[0]}")
print(f"Total unique session_id rows: {df["ga_session_id"].nunique()}")

Total session_id rows: 93280
Total unique session_id rows: 48083


In [13]:
# correcting data types
df["date"] = pd.to_datetime(df["date"])
df["ga_session_id"] = df["ga_session_id"].astype("string")
df["channel"] = df["channel"].astype("category")
df["continent"] = df["continent"].astype("category")
df["device"] = df["device"].astype("category")
df["country"] = df["country"].astype("category")
df["test"] = df["test"].astype("category")
df["test_group"] = df["test_group"].astype("category")
df["event_name"] = df["event_name"].astype("category")

df.dtypes

date             datetime64[us]
ga_session_id            string
country                category
device                 category
continent              category
channel                category
test                   category
test_group             category
event_name             category
value                     int64
dtype: object

In [14]:
# cheking date range of the dataset
print(f"Start Date: {df["date"].min().date()}, Finish Date: {df["date"].max().date()}")
print(f"Days Range of the Dataset: {(df["date"].max() - df["date"].min()).days}")

Start Date: 2020-11-01, Finish Date: 2021-01-27
Days Range of the Dataset: 87


In [15]:
# cheking number of unique values per column
categorical_columns = [
 'country',
 'device',
 'continent',
 'channel',
 'test',
 'test_group',
 'event_name',
 'value'
]

for column in categorical_columns:
  print(f"Number of unique values in {column} column: {df[column].nunique()}")

Number of unique values in country column: 85
Number of unique values in device column: 3
Number of unique values in continent column: 6
Number of unique values in channel column: 5
Number of unique values in test column: 4
Number of unique values in test_group column: 2
Number of unique values in event_name column: 19
Number of unique values in value column: 97


In [16]:
# inspecting value column
df["value"].describe()

print(f"Min value: {df["value"].min()}")
print(f"Median value: {df["value"].median()}")
print(f"Max value: {df["value"].max()}")
print(f"Number of values higher than 1: {(df["value"] > 1).sum()}")

Min value: 1
Median value: 1.0
Max value: 223
Number of values higher than 1: 22345


In [17]:
# visual inspection of 10 highest values in value column
df[df["value"] > 1].sort_values(by="value", ascending=False).head(10)

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
42641,2020-12-09,6649586044,Finland,mobile,Europe,Social Search,4,1,user_engagement,223
49391,2020-12-09,6649586044,Finland,mobile,Europe,Social Search,4,1,view_item,189
68262,2021-01-20,9321185558,Brazil,desktop,Americas,Organic Search,4,1,page_view,120
85140,2020-12-19,1174063066,Canada,desktop,Americas,Direct,4,1,page_view,116
44861,2020-11-22,3428090267,Peru,desktop,Americas,Organic Search,1,1,user_engagement,116
81948,2020-11-09,1075293762,Canada,desktop,Americas,Organic Search,2,1,user_engagement,115
64339,2020-11-22,548130028,Australia,desktop,Oceania,Organic Search,1,1,user_engagement,115
89873,2020-11-11,3331939525,Canada,mobile,Americas,Direct,2,1,page_view,115
89874,2020-11-12,4079937900,Canada,mobile,Americas,Paid Search,2,2,user_engagement,115
22399,2020-11-22,1874448011,Norway,mobile,Europe,Organic Search,2,2,page_view,115


Comments:

* The initial dataset contains over 3.2M rows and 334K unique sessions.
* Unique sessions account for approximately 10% of the total number of rows, reflecting the event-level structure of the dataset.
* The dataset covers 87 days, from November 1, 2020, to January 27, 2021.
* Over 808K rows have a value greater than 1, indicating that certain events occurred multiple times within individual sessions.

# 3. Statistical Significance

## 3.1 Creating DataFrame without Segmentation

In [18]:
# Checking statistical significance of metrics using z-proportions test

def metric_stat_significance(df):

    metrics = [
        "add_payment_info",
        "add_shipping_info",
        "begin_checkout",
        "new_account"
    ]

    df_total = pd.DataFrame(
        columns=[
            "test",
            "metric",
            "dimension",
            "segment",
            "numerator_type",
            "denominator_type",
            "numerator_count_control",
            "denominator_count_control",
            "conversion_rate_control",
            "numerator_count_experimental",
            "denominator_count_experimental",
            "conversion_rate_experimental",
            "metric_change (%)",
            "z_stat",
            "p_value",
            "significant?"
        ]
    )

    for test in df["test"].unique():

        control_total = (
            df.loc[
                (df["test"] == test) &
                (df["test_group"] == 1) &
                (df["event_name"] == "session"),
                "value"
            ]
            .sum()
        )

        experimental_total = (
            df.loc[
                (df["test"] == test) &
                (df["test_group"] == 2) &
                (df["event_name"] == "session"),
                "value"
            ]
            .sum()
        )

        for metric in metrics:

            control_df = df.loc[
                (df["test"] == test) &
                (df["test_group"] == 1) &
                (df["event_name"] == metric)
            ]

            experimental_df = df.loc[
                (df["test"] == test) &
                (df["test_group"] == 2) &
                (df["event_name"] == metric)
            ]

            control_successes = control_df["value"].sum()
            experimental_successes = experimental_df["value"].sum()

            conversion_rate_control = (
                control_successes / control_total
            )

            conversion_rate_experimental = (
                experimental_successes / experimental_total
            )

            z_stat, p_value = sm.stats.proportions_ztest(
                [experimental_successes, control_successes],
                [experimental_total, control_total]
            )

            metric_change = (
                (conversion_rate_experimental - conversion_rate_control)
                / conversion_rate_control
                * 100
            )

            significant_results = p_value < 0.05

            # filling up the dataframe rows
            df_total.loc[len(df_total)] = [
                test,
                metric + " / session",
                "total",
                "all",
                metric,
                "session",
                control_successes,
                control_total,
                conversion_rate_control,
                experimental_successes,
                experimental_total,
                conversion_rate_experimental,
                metric_change,
                z_stat,
                p_value,
                significant_results
            ]

    return df_total


df_total = metric_stat_significance(df)

## 3.2 Creating DataFrame with Segmentation

In [19]:
# Checking statistical significance of metrics across all segments using z-proportions test

def metric_stat_significance_segment(df):

    metrics = [
        "add_payment_info",
        "add_shipping_info",
        "begin_checkout",
        "new_account"
    ]

    dimensions = [
        "continent",
        "country",
        "device",
        "channel"
    ]

    # Total number of sessions for each test and group
    session_counts = (
        df.loc[df["event_name"] == "session"].groupby(
            ["test", "test_group"],
            observed=True
        )["value"]
        .sum()
        .to_dict()
    )

    # Empty result DataFrame
    df_segment = pd.DataFrame(
        columns=[
            "test",
            "metric",
            "dimension",
            "segment",
            "numerator_type",
            "denominator_type",
            "numerator_count_control",
            "denominator_count_control",
            "conversion_rate_control",
            "numerator_count_experimental",
            "denominator_count_experimental",
            "conversion_rate_experimental",
            "metric_change (%)",
            "z_stat",
            "p_value",
            "significant?"
        ]
    )

    # Calculate results separately for each dimension
    for dimension in dimensions:

        aggregated = (
            df.groupby(
                [
                    "test",
                    "test_group",
                    "event_name",
                    dimension
                ],
                observed=True,
                dropna=False
            )["value"]
            .sum()
            .reset_index()
        )

        for test in df["test"].unique():

            # Denominator is NOT segmented
            control_total = session_counts.get((test, 1), 0)
            experimental_total = session_counts.get((test, 2), 0)

            # Skip test if one of the groups has no sessions
            if control_total == 0 or experimental_total == 0:
                continue

            for metric in metrics:

                control_df = aggregated.loc[
                    (aggregated["test"] == test) &
                    (aggregated["test_group"] == 1) &
                    (aggregated["event_name"] == metric)
                ]

                experimental_df = aggregated.loc[
                    (aggregated["test"] == test) &
                    (aggregated["test_group"] == 2) &
                    (aggregated["event_name"] == metric)
                ]

                # All segments that exist in either group
                segments = (
                    set(control_df[dimension].dropna())
                    |
                    set(experimental_df[dimension].dropna())
                )

                for segment in segments:

                    control_segment = control_df.loc[
                        control_df[dimension] == segment,
                        "value"
                    ]

                    experimental_segment = experimental_df.loc[
                        experimental_df[dimension] == segment,
                        "value"
                    ]

                    control_successes = control_segment.sum()
                    experimental_successes = experimental_segment.sum()

                    conversion_rate_control = (
                        control_successes / control_total
                    )

                    conversion_rate_experimental = (
                        experimental_successes / experimental_total
                    )

                    # Avoid 0 / 0 inside proportions_ztest
                    if (
                        control_successes == 0
                        and experimental_successes == 0
                    ):
                        z_stat = np.nan
                        p_value = np.nan
                    else:
                        z_stat, p_value = sm.stats.proportions_ztest(
                            [
                                experimental_successes,
                                control_successes
                            ],
                            [
                                experimental_total,
                                control_total
                            ]
                        )

                    # Avoid division by zero in relative change
                    if conversion_rate_control == 0:
                        metric_change = np.nan
                    else:
                        metric_change = (
                            (
                                conversion_rate_experimental
                                - conversion_rate_control
                            )
                            / conversion_rate_control
                            * 100
                        )

                    # If p-value is NaN, significance is also undefined
                    if pd.isna(p_value):
                        significant_results = np.nan
                    else:
                        significant_results = p_value < 0.05

                    # Filling in the dataframe rows
                    df_segment.loc[len(df_segment)] = [
                        test,
                        metric + " / session",
                        dimension,
                        segment,
                        metric,
                        "session",
                        control_successes,
                        control_total,
                        conversion_rate_control,
                        experimental_successes,
                        experimental_total,
                        conversion_rate_experimental,
                        metric_change,
                        z_stat,
                        p_value,
                        significant_results
                    ]

    return df_segment

df_segment = metric_stat_significance_segment(df)


## 3.3 Creating & Downloading the Final DataFrame

In [20]:
# Concating both dataframes
final_df = pd.concat(
        [df_total, df_segment],
        ignore_index=True
        )

final_df.head()

,test,metric,dimension,segment,numerator_type,denominator_type,numerator_count_control,denominator_count_control,conversion_rate_control,numerator_count_experimental,denominator_count_experimental,conversion_rate_experimental,metric_change (%),z_stat,p_value,significant?
0,2,add_payment_info / session,total,all,add_payment_info,session,73,1736,0.042051,57,1711,0.033314,-20.776923,-1.346265,0.178217,False
1,2,add_shipping_info / session,total,all,add_shipping_info,session,103,1736,0.059332,98,1711,0.057276,-3.464164,-0.257474,0.796813,False
2,2,begin_checkout / session,total,all,begin_checkout,session,132,1736,0.076037,137,1711,0.080070,5.304359,0.441392,0.658929,False
3,2,new_account / session,total,all,new_account,session,144,1736,0.082949,132,1711,0.077148,-6.993961,-0.627486,0.530341,False
4,1,add_payment_info / session,total,all,add_payment_info,session,67,1563,0.042866,38,1631,0.023299,-45.648215,-3.100344,0.001933,True


In [ ]:
# Extracting data source for Tableau

# final_df.to_csv("significance_df_tableau.csv", index=False)

[Dataset URL](https://drive.google.com/file/d/1_Gc0DxW0ZzD7a3WniAZ8UQggJIwE1cPy/view?usp=drive_link)

[Tableau Dashboard URL](https://public.tableau.com/views/StatisticalSignificanceofABTestMetrics/StatisticalSignificanceAnalysisforABtest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

**Comments:**
* First DataFrame calculates all metric in total, with only 2 dimensions: test and test_group.
* Second DataFrame calculates all metrics in all possible segments.
* Conversion rate is calculated as the total number of metric events divided by the total number of sessions within the corresponding test and test group. For segmented analysis, the numerator is filtered by the selected segment, while the denominator remains the total number of sessions in the test group.

# 4. Conclusions


**Statistical Significance - Key Findings:**

* Test 1 showed the strongest overall positive results. add_payment_info and add_shipping_info demonstrated statistically significant improvements in Europe, while begin_checkout improved across most continents. Africa was an exception, with a statistically significant decrease of approximately 40% in begin_checkout and Americas with practically insignificant positive change.

* new_account consistently showed limited evidence of improvement. Across Tests 1, 3, and 4, no statistically significant positive changes were observed across the analyzed dimensions. In Test 4, the overall new_account metric also showed a statistically significant decrease of 3.3%.

* Test 2 did not produce statistically significant overall changes in any of the analyzed metrics. Significant effects were observed only in a limited number of countries and continents, including statistically significant improvements in add_payment_info and begin_checkout in Indonesia.

* begin_checkout showed heterogeneous results across tests and segments. While positive changes were observed in several segments, statistically significant negative changes occurred at the overall level in Test 3 (−3.3%) and Test 4 (−2.3%).

* Indonesia repeatedly showed statistically significant positive changes, particularly in begin_checkout. The metric increased by approximately 45% in Test 3 and 60.7% in Test 4. Singapore also frequently showed positive changes in segments where Indonesia improved, although this pattern was not consistent across every test or metric.

* Test 4 showed contrasting results between overall and country-level performance. While begin_checkout and new_account decreased significantly overall, Indonesia showed substantial positive changes in begin_checkout (+60.7%) and add_shipping_info (+76.8%).

* The segment-level analysis indicates that the observed effects were not uniform across geographic segments. Several statistically significant changes were concentrated in specific countries or continents rather than consistently appearing across the entire user population.